In [ ]:
import numpy as np
import os
from os import path, listdir
import pickle

# large3 = np.load("data_npz/large_tight_bound.npz")
# print(large3.files)
# raw = large3["raw"]
# print(raw.shape)
# print(raw)
# print(large3["priors_samples"].shape)


In [ ]:
sweep_results_dir = "sweep_results"

files= listdir(sweep_results_dir)

for file in files:
    if file.endswith(".pkl"):
        with open(path.join(sweep_results_dir, file), "rb") as f:
            data = pickle.load(f)
            
        print(f"File: {file}")
        print(f"Data keys: {data.keys()}")
        for key, value in data.items():
            if key == "lambda_p": continue
            sweep_vals = value["sweep_values"]
            median_means = value["median_means"]
            lower_bounds = value["lower_bounds"]
            upper_bounds = value["upper_bounds"]
            cov_90 = value["coverage_90"]
        
            print(f"  {key}:")
            print(f"    Sweep values: {sweep_vals}")
            print(f"    Median means: {median_means}")
            print(f"    90% CI lower bounds: {lower_bounds}")
            print(f"    90% CI upper bounds: {upper_bounds}")
            print(f"    90% CI coverage: {cov_90}")

In [ ]:
recovery_results_dir = "cov_results"

files = listdir(recovery_results_dir)
for file in files:
    if file.endswith(".pkl"):
        with open(path.join(recovery_results_dir, file), "rb") as f:
            data = pickle.load(f)
            
        print(f"File: {file}")
        print(f"Data keys: {data.keys()}")
        
        coverage_results = data["coverage_results"]
        mean_median = data["mean_median"]
        std_median = data["std_median"]
        mean_interval_width = data["mean_interval_width"][0]
        levels = data["levels"]

        medians_std = zip(mean_median, std_median)

        coverage_results = coverage_results[f"{(1 - 2 * np.float32(0.025)):3.0%}"]

        print(f"    Coverage results: {' | '.join(f'{x:2.2%}' for x in coverage_results)}")
        print(f"    Mean of medians: {' & '.join(f'{mean:.3f} ({std:.3f})' for mean, std in medians_std)}")
        print(f"    Std of medians: {np.array2string(std_median, precision=3, suppress_small=False)}")
        print(f"    Mean interval width: {' & '.join(f'{width:.3f}' for width in mean_interval_width)}")
        print(f"    Levels: {np.array2string(levels, precision=3, suppress_small=False)}")


In [ ]:
from sbi.examples.minimal import simple
from sbi.diagnostics import run_sbc, check_sbc
from sbi.utils import BoxUniform

import torch

param_dim = 3
reduce_fns = [eval(f"lambda theta, x: theta[:, {i}]") for i in range(param_dim)]

x_o = torch.ones(3)

prior = BoxUniform(low=torch.ones(3) * (-2), high=torch.ones(3) * 2)

post = simple()
post.set_default_x(x_o)

reduce_fns.append(lambda theta, x: -post.log_prob(theta, x))

theta = prior.sample((200,))

xs = torch.randn(200, 3)

ranks, dap_samples, posterior_samples = run_sbc(theta, xs, post, reduce_fns=reduce_fns)
print(ranks.shape)
print(dap_samples.shape)
print(posterior_samples.shape)


In [8]:
sbc_res_dir = "sbc_res"

files = listdir(sbc_res_dir)
for file in files:
    if file.endswith(".pkl"):
        with open(path.join(sbc_res_dir, file), "rb") as f:
            data = pickle.load(f)
            
        print(f"File: {file}")
        print(f"Data keys: {data.keys()}")
        
        for key, value in data.items():
            print(f"  {key}:")
            print(f"    Ranks shape: {value['ranks'].shape}")
            print(f"    DAP samples shape: {value['dap_samples'].shape}")

            mean_rank = value["ranks"].float().mean(dim=0)
            std_rank = value["ranks"].float().std(dim=0)
            print(f"    Mean ranks: {" & ".join(f"{mean:.2f} ({std:.2f})" for mean, std in zip(mean_rank, std_rank))}")

            ### rank appx discrete uniform 0, 1000 -> var (r) = (N + 1) (N - 1) / 12
            sd_r = ((1000 + 1) * (1000 - 1) / 12) ** 0.5
            sde_r = sd_r / (1000 ** 0.5)
            z_scores = (mean_rank - 500) / sde_r
            print(f"    Z-scores: {' & '.join(f'{z:.2f}' for z in z_scores)}")

File: sbc_results_IT_prior_1000_post_1000_20260622_162726_seed_0.pkl
Data keys: dict_keys(['npe_simple_hierarchical', 'npe_hierarchical', 'npe_seq_multivariate', 'npe_m,s,min_gr,max_gr,ac1,ac2,ac3,q25_gr,q50_gr,q75_gr'])
  npe_simple_hierarchical:
    Ranks shape: torch.Size([1000, 3])
    DAP samples shape: torch.Size([1000, 3])
    Mean ranks: 531.15 (298.20) & 501.33 (288.65) & 516.88 (315.10)
    Z-scores: 3.41 & 0.15 & 1.85
  npe_hierarchical:
    Ranks shape: torch.Size([1000, 3])
    DAP samples shape: torch.Size([1000, 3])
    Mean ranks: 490.69 (295.57) & 506.24 (293.10) & 541.99 (292.74)
    Z-scores: -1.02 & 0.68 & 4.60
  npe_seq_multivariate:
    Ranks shape: torch.Size([1000, 3])
    DAP samples shape: torch.Size([1000, 3])
    Mean ranks: 507.82 (289.78) & 510.05 (291.42) & 522.64 (284.11)
    Z-scores: 0.86 & 1.10 & 2.48
  npe_m,s,min_gr,max_gr,ac1,ac2,ac3,q25_gr,q50_gr,q75_gr:
    Ranks shape: torch.Size([1000, 3])
    DAP samples shape: torch.Size([1000, 3])
    Mean r